# Módulo 3 — NCF: Sistema de Recomendación de Destinos
**Datos reales**: `amanmehra23/travel-recommendation-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, sys
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cpu')
BATCH_SIZE=128; EPOCHS=50; LR=0.001; EMB_DIM=32
BASE_DIR=Path('.'); MODELS_DIR=BASE_DIR/'models'; MODELS_DIR.mkdir(exist_ok=True)
print(f"Config: EPOCHS={EPOCHS} | LR={LR} | EMB_DIM={EMB_DIM} | device={device}")

## 1. Carga del dataset real de Kaggle

In [ ]:
import kagglehub
print("Descargando dataset desde Kaggle...")
try:
    raw_path = Path(kagglehub.dataset_download("amanmehra23/travel-recommendation-dataset"))
except Exception as e:
    sys.exit(f"ERROR: {e}\nConfigura ~/.kaggle/kaggle.json")

csvs = {p.stem.lower(): p for p in raw_path.rglob("*.csv")}
reviews_path = next((p for k,p in csvs.items() if "review" in k), None)
dest_path    = next((p for k,p in csvs.items() if "destination" in k), None)
assert reviews_path and dest_path, f"CSVs no encontrados: {list(csvs.values())}"

df_reviews = pd.read_csv(reviews_path)
df_dest    = pd.read_csv(dest_path)
print(f"Reviews: {df_reviews.shape}  Destinations: {df_dest.shape}")
print("Reviews columnas:", list(df_reviews.columns))
print(df_reviews.head(3))

## 2. Preprocesamiento

In [ ]:
# Label positivo: Rating >= 3
rating_col = next((c for c in df_reviews.columns if 'rating' in c.lower() or 'score' in c.lower()), None)
assert rating_col, f"No hay columna de rating. Columnas: {list(df_reviews.columns)}"
print(f"Columna rating: {rating_col}")

df_reviews['label'] = (df_reviews[rating_col] >= 3).astype(int)
print(f"Positivos: {df_reviews['label'].sum()} ({df_reviews['label'].mean()*100:.1f}%)")

user_col = next(c for c in df_reviews.columns if 'user' in c.lower())
item_col = next(c for c in df_reviews.columns if 'dest' in c.lower() or 'item' in c.lower())
print(f"User col: {user_col} | Item col: {item_col}")

# Filtrar
uc = df_reviews[user_col].value_counts()
ic = df_reviews[item_col].value_counts()
df = df_reviews[df_reviews[user_col].isin(uc[uc>=3].index) &
                df_reviews[item_col].isin(ic[ic>=5].index)].copy()
print(f"Post-filtrado: {df.shape} | usuarios={df[user_col].nunique()} | items={df[item_col].nunique()}")

ue = LabelEncoder(); ie = LabelEncoder()
df['user_enc'] = ue.fit_transform(df[user_col])
df['item_enc'] = ie.fit_transform(df[item_col])
n_users = df['user_enc'].nunique(); n_items = df['item_enc'].nunique()
print(f"Encoded: {n_users} usuarios | {n_items} items")

## 3. Negative sampling y split

In [ ]:
all_items = set(range(n_items))
user_visited = df.groupby('user_enc')['item_enc'].apply(set).to_dict()
positives = df[df['label']==1].copy()

neg_rows=[]
for _,row in positives.iterrows():
    u=int(row['user_enc']); vis=user_visited[u]
    cands=list(all_items-vis)
    neg=np.random.choice(cands if len(cands)>=4 else list(all_items), size=4, replace=False)
    for ni in neg: neg_rows.append({'user_enc':u,'item_enc':int(ni),'label':0})

df_pos = positives[['user_enc','item_enc','label']].copy()
df_all = pd.concat([df_pos, pd.DataFrame(neg_rows)], ignore_index=True).sample(frac=1,random_state=42)

# Split 80/20 por usuario
train_r,test_r=[],[]
for uid,grp in df_pos.groupby('user_enc'):
    n_te=max(1,int(len(grp)*0.2)); test_r.append(grp.iloc[-n_te:]); train_r.append(grp.iloc[:-n_te])
df_test=pd.concat(test_r,ignore_index=True); df_train_pos=pd.concat(train_r,ignore_index=True)

neg2=[]
for _,row in df_train_pos.iterrows():
    u=int(row['user_enc']); vis=user_visited[u]; cands=list(all_items-vis)
    neg=np.random.choice(cands if len(cands)>=4 else list(all_items), size=4, replace=False)
    for ni in neg: neg2.append({'user_enc':u,'item_enc':int(ni),'label':0})
df_train=pd.concat([df_train_pos,pd.DataFrame(neg2)],ignore_index=True).sample(frac=1,random_state=42)
print(f"Train: {len(df_train)} | Test positivos: {len(df_test)}")

## 4. Arquitectura NCF

In [ ]:
class NCF(nn.Module):
    def __init__(self,n_users,n_items,emb_dim=32):
        super().__init__()
        self.user_emb=nn.Embedding(n_users,emb_dim)
        self.item_emb=nn.Embedding(n_items,emb_dim)
        self.mlp=nn.Sequential(
            nn.Linear(emb_dim*2,128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
        nn.init.normal_(self.user_emb.weight,std=0.01)
        nn.init.normal_(self.item_emb.weight,std=0.01)
        for l in self.mlp:
            if isinstance(l,nn.Linear): nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    def forward(self,u,i):
        return self.mlp(torch.cat([self.user_emb(u),self.item_emb(i)],dim=1)).squeeze()

class IntDS(Dataset):
    def __init__(self,df):
        self.u=torch.tensor(df['user_enc'].values,dtype=torch.long)
        self.i=torch.tensor(df['item_enc'].values,dtype=torch.long)
        self.y=torch.tensor(df['label'].values,dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.u[i],self.i[i],self.y[i]

print(f"NCF: {n_users} users | {n_items} items | emb_dim={EMB_DIM}")

## 5. Entrenamiento

In [ ]:
model=NCF(n_users,n_items,EMB_DIM).to(device)
opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=1e-5)
crit=nn.BCELoss()
loader=DataLoader(IntDS(df_train),batch_size=BATCH_SIZE,shuffle=True)

loss_hist=[]
for ep in range(1,EPOCHS+1):
    model.train(); el=0.0
    for u,i,y in loader:
        u,i,y=u.to(device),i.to(device),y.to(device)
        opt.zero_grad(); loss=crit(model(u,i),y); loss.backward(); opt.step()
        el+=loss.item()*len(u)
    el/=len(loader.dataset); loss_hist.append(el)
    if ep%10==0 or ep==1: print(f"Epoch {ep:3d}/{EPOCHS} | Loss: {el:.4f}")

plt.figure(figsize=(10,4))
plt.plot(loss_hist,color='#2C5282',linewidth=2)
plt.title("Curva de pérdida NCF"); plt.xlabel("Época"); plt.ylabel("BCELoss")
plt.grid(alpha=0.3); plt.savefig('fig_ncf_loss.png',dpi=120,bbox_inches='tight'); plt.show()

## 6. Evaluación Precision@5, Recall@5, NDCG@10

In [ ]:
model.eval()
test_pos={u:set(g['item_enc'].values) for u,g in df_test.groupby('user_enc')}
train_pos={u:set(g['item_enc'].values) for u,g in df_train_pos.groupby('user_enc')}
all_items_list=list(range(n_items))

prec5,rec5,ndcg10=[],[],[]
for u_enc,relevant in test_pos.items():
    excl=train_pos.get(u_enc,set())
    cands=[i for i in all_items_list if i not in excl]
    if not cands: continue
    u_t=torch.tensor([u_enc]*len(cands),dtype=torch.long)
    i_t=torch.tensor(cands,dtype=torch.long)
    with torch.no_grad(): sc=model(u_t,i_t).numpy()
    top5=[cands[i] for i in np.argsort(sc)[::-1][:5]]
    top10=[cands[i] for i in np.argsort(sc)[::-1][:10]]
    h5=len(set(top5)&relevant)
    prec5.append(h5/5); rec5.append(h5/max(len(relevant),1))
    dcg=sum(1.0/np.log2(r+2) for r,it in enumerate(top10) if it in relevant)
    idcg=sum(1.0/np.log2(r+2) for r in range(min(len(relevant),10)))
    ndcg10.append(dcg/idcg if idcg>0 else 0.0)

print(f"Precision@5: {np.mean(prec5):.4f}")
print(f"Recall@5:    {np.mean(rec5):.4f}")
print(f"NDCG@10:     {np.mean(ndcg10):.4f}")
metrics_ncf={'Precision@5':float(np.mean(prec5)),'Recall@5':float(np.mean(rec5)),'NDCG@10':float(np.mean(ndcg10))}

## 7. Ejemplos de recomendaciones para 5 usuarios

In [ ]:
# Construir catálogo de items
if "Name" in df_dest.columns:
    dest_cat_df = df_dest.drop_duplicates(subset=[item_col] if item_col in df_dest.columns else ["DestinationID"])
    catalog = {}
    for _,row in dest_cat_df.iterrows():
        did = row.get("DestinationID", row.get(item_col,"?"))
        if did in ie.classes_:
            enc = ie.transform([did])[0]
            catalog[int(enc)] = {
                'name': str(row.get("Name","?")),
                'type': str(row.get("Type","N/A")),
                'state': str(row.get("State","N/A")),
                'popularity': float(row.get("Popularity",0)),
                'best_time': str(row.get("BestTimeToVisit","N/A")),
            }
else:
    catalog={enc:{'name':str(ie.inverse_transform([enc])[0]),'type':'N/A','state':'N/A','popularity':0,'best_time':'N/A'}
             for enc in range(n_items)}

sample_users=list(test_pos.keys())[:5]
for u_enc in sample_users:
    u_id = ue.inverse_transform([u_enc])[0]
    excl = train_pos.get(u_enc,set())
    cands=[i for i in range(n_items) if i not in excl]
    u_t=torch.tensor([u_enc]*len(cands),dtype=torch.long)
    i_t=torch.tensor(cands,dtype=torch.long)
    with torch.no_grad(): sc=model(u_t,i_t).numpy()
    top5=[cands[i] for i in np.argsort(sc)[::-1][:5]]
    print(f"\nUsuario {u_id} — Top 5 recomendaciones:")
    for rank,ie_ in enumerate(top5,1):
        info=catalog.get(ie_,{'name':f'Item {ie_}','type':'?','state':'?'})
        print(f"  {rank}. {info['name']} ({info['type']}, {info['state']}) — score={sc[cands.index(ie_)]:.4f}")

## 8. Análisis PCA de embeddings

In [ ]:
with torch.no_grad():
    item_embs = model.item_emb.weight.numpy()
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(item_embs)
types=[catalog.get(i,{}).get('type','?') for i in range(n_items)]
unique_types=list(set(types)); colors=plt.cm.tab10(np.linspace(0,1,len(unique_types)))
fig,ax=plt.subplots(figsize=(10,7))
for t,c in zip(unique_types,colors):
    idx=[i for i,tp in enumerate(types) if tp==t]
    ax.scatter(coords[idx,0],coords[idx,1],label=t,alpha=0.6,s=30,color=c)
ax.legend(fontsize=8,ncol=2); ax.set_title("PCA de embeddings de destinos (2D)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout()
plt.savefig('fig_embeddings_pca.png',dpi=120,bbox_inches='tight'); plt.show()

## 9. Guardar artefactos

In [ ]:
# Construir dest_catalog para webapp
dest_catalog=[]
for enc,(item_id) in enumerate(ie.classes_):
    info=catalog.get(enc,{})
    dest_catalog.append({
        'DestinationID': int(item_id),
        'Name': info.get('name',str(item_id)),
        'Type': info.get('type','N/A'),
        'State': info.get('state','N/A'),
        'Popularity': info.get('popularity',0),
        'BestTimeToVisit': info.get('best_time','N/A'),
        'description': f"Explora {info.get('name',item_id)} en {info.get('state','India')}."
    })

torch.save(model.state_dict(), MODELS_DIR/'ncf_model.pt')
ncf_meta={'user_encoder':ue,'item_encoder':ie,'n_users':n_users,'n_items':n_items,
           'dest_catalog':dest_catalog,'metrics':metrics_ncf}
with open(MODELS_DIR/'ncf_metadata.pkl','wb') as f: pickle.dump(ncf_meta,f)

import shutil
webapp_models=Path('../webapp/models'); webapp_models.mkdir(exist_ok=True)
for fname in ['ncf_model.pt','ncf_metadata.pkl']:
    shutil.copy(MODELS_DIR/fname, webapp_models/fname)
print("[OK] Artefactos NCF guardados.")
print(f"Metrics: {metrics_ncf}")